# Test synchronisation Docs → Grist
Notebook de test pour valider chaque étape avant la synchro complète.

In [1]:
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

True

## 1. Connexion au client Docs
Renseigne `DOCS_SESSION_ID` et `DOCS_CSRF_TOKEN` dans le fichier `.env` avant de lancer cette cellule.

In [2]:
import os
from docs_client import DocsClient

docs = DocsClient(
    base_url='https://docs.numerique.gouv.fr',
    session_id=os.environ.get('DOCS_SESSION_ID'),
    csrf_token=os.environ.get('DOCS_CSRF_TOKEN'),
)
print('DocsClient initialisé.')

DocsClient initialisé.


## 2. Récupération du tree
Remplace l'URL ci-dessous par celle de ton document racine.

In [3]:
ROOT_URL = 'https://docs.numerique.gouv.fr/docs/21bb0cdd-638f-4a50-8f3a-03616716771e/'

doc_id = DocsClient.extract_doc_id(ROOT_URL)
tree = docs.get_tree(doc_id)

print(f"Racine : {tree['title']}")
print(f"Enfants directs : {len(tree.get('children', []))}")

Racine : Cartographie Stratégique du Bruit - Guide Méthodologique
Enfants directs : 13


## 3. Aplatissement du tree en records Grist

In [5]:
from importlib import reload
import docs_client
reload(docs_client)
from docs_client import DocsClient

records = docs.flatten_tree(tree, 'https://docs.numerique.gouv.fr', is_root=True)

print(f"\n{len(records)} chapitre(s) extrait(s).")

[racine] Cartographie Stratégique du Bruit - Guide Méthodologique
  [1] 🌟 Guide
    [1.1] 📜 Rapportage
    [1.2] 🛣️ GITT
    [1.3] 🗺️ CBS
    [1.4] 🔠 Glossaire
    [1.5] ℹ️ À propos du guide
  [2] 🛠️ Outils
    [2.1] 🔊 NoiseModelling
    [2.2] 💽 Bases de données
    [2.3] 🖥️ Serveur de calcul
  [3] 🏘️ ​🔴 Bâtiments
    [3.1] 🙎‍♂️ ​🔴 Affectation des populations
    [3.2] 🧑‍🔧 Implémentation - Bâti
  [4] 📍 Récepteurs
    [4.1] 👩‍👦 ​🔵 Décompte de population
    [4.2] 🎨 Visuel - isosurfaces
    [4.3] 🧑‍🔧 ​🔵 Implémentation - Récepteurs
  [5] 🚙 Routier
    [5.1] 🌐 ​🔴 Géométrie
    [5.2] 📊 ​🔴 Débits
    [5.3] ⏩ ​🔴 Vitesses
    [5.4] 🛣️ Revêtements
    [5.5] ❄️ Pneus neige
    [5.6] 🧑‍🔧 Implémentation - Route
  [6] 🚅 Ferroviaire
    [6.1] 🚇 ​🔴 Tramway, Métro
    [6.2] 📊 ​🔵 Débits
    [6.3] ⏩ ​🔵 Vitesses
    [6.4] 🛤️ Plateforme
    [6.5] 🧑‍🔧 Implémentation - Fer
  [7] 🧱 ​🔴 Écrans acoustiques
    [7.1] 🧑‍🔧 Implémentation - Écrans
  [8] 🌉 ​🔵 Ponts et tunnels
  [9] 🏞️ Occupation du sol
    [9.1] 🧑‍🔧

## 4. Aperçu des records

In [7]:
import pandas as pd

df = pd.DataFrame([r['fields'] for r in records])
df[['numero', 'niveau', 'ordre', 'titre', 'url', 'contenu']].head(20)

,numero,niveau,ordre,titre,url,contenu
0,1,1,1,🌟 Guide,https://docs.numerique.gouv.fr/docs/242c7d7f-b...,Ce guide méthodologique est construit autour d...
1,1.1,2,0,📜 Rapportage,https://docs.numerique.gouv.fr/docs/d3521e51-f...,## Indicateurs\n\nLa liste des indicateurs att...
2,1.2,2,0,🛣️ GITT,https://docs.numerique.gouv.fr/docs/e85bb5f4-5...,"À chaque nouvelle échéance, l'état est en char..."
3,1.3,2,0,🗺️ CBS,https://docs.numerique.gouv.fr/docs/3c2d989c-4...,"Au regard de la réglementation, il existe 4 ty..."
4,1.4,2,0,🔠 Glossaire,https://docs.numerique.gouv.fr/docs/6b70ef38-7...,* **CBS** : Carte de Bruit Stratégique\n\n* **...
5,1.5,2,0,ℹ️ À propos du guide,https://docs.numerique.gouv.fr/docs/bae48920-3...,## Auteurs :\n\n* Gwendall Petit ([UMRAE](http...
6,2,1,3,🛠️ Outils,https://docs.numerique.gouv.fr/docs/1755bc0c-5...,Dans cette section nous listons les outils gra...
7,2.1,2,0,🔊 NoiseModelling,https://docs.numerique.gouv.fr/docs/9ee6bf02-1...,![\_images/NoiseModelling\_banner.png](https:/...
8,2.2,2,0,💽 Bases de données,https://docs.numerique.gouv.fr/docs/17dcf47e-1...,## PostGreSQL / PostGIS\n\nL'ensemble des donn...
9,2.3,2,0,🖥️ Serveur de calcul,https://docs.numerique.gouv.fr/docs/adaecfd6-6...,La réalisation des cartes de bruits et le calc...


## 5. Envoi vers Grist
⚠️ Lance cette cellule uniquement quand les records semblent corrects.

In [ ]:
from grist_client import GristClient

grist = GristClient()
result = grist.add_records('Chapitres', records)
print(f"Enregistrements ajoutés : {result}")